# CatBoost - Hyperparameter Sweeps

## Setup and Imports

In [1]:
import sys

sys.path.append("..")

import wandb
import dotenv

from src.api.run import sweep_catboost_ensemble
from src.api.sweep import wandb_sweep

D:\git\code\.venv\Lib\site-packages\pydantic\_internal\_generate_schema.py:2249: UnsupportedFieldAttributeWarning: The 'repr' attribute with value False was provided to the `Field()` function, which has no effect in the context it was used. 'repr' is field-specific metadata, and can only be attached to a model field using `Annotated` metadata or by assignment. This may have happened because an `Annotated` type alias using the `type` statement was used, or if the `Field()` function was attached to a single member of a union type.
  warnings.warn(
D:\git\code\.venv\Lib\site-packages\pydantic\_internal\_generate_schema.py:2249: UnsupportedFieldAttributeWarning: The 'frozen' attribute with value True was provided to the `Field()` function, which has no effect in the context it was used. 'frozen' is field-specific metadata, and can only be attached to a model field using `Annotated` metadata or by assignment. This may have happened because an `Annotated` type alias using the `type` statemen

In [2]:
dotenv.load_dotenv()
wandb.login()

wandb: Currently logged in as: schurtenberger-david (david-schurtenberger) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


True

## Sweep Configuration

In [3]:
max_runs = 100
sweep_config = {
    "name": "CatBoost Ensemble",
    "method": "bayes",
    "metric": {"name": "cv_brier", "goal": "minimize"},
    "parameters": {
        "catboost_config": {
            "parameters": {
                "iterations": {"distribution": "int_uniform", "min": 200, "max": 1000},
                "learning_rate": {"distribution": "log_uniform_values", "min": 0.001, "max": 0.3},
                "depth": {"distribution": "int_uniform", "min": 3, "max": 10},
                "l2_leaf_reg": {"distribution": "log_uniform_values", "min": 0.1, "max": 10.0},
                "random_strength": {"distribution": "uniform", "min": 0.0, "max": 5.0},
                "bagging_temperature": {"distribution": "uniform", "min": 0.0, "max": 5.0},
                "subsample": {"distribution": "uniform", "min": 0.5, "max": 1.0},
                "border_count": {"values": [32, 64, 128, 254]},
                "grow_policy": {"values": ["SymmetricTree", "Lossguide", "Depthwise"]},
                "min_data_in_leaf": {"distribution": "int_uniform", "min": 1, "max": 10},
            },
        },
        "run_config": {
            "parameters": {
                "valid_season": {"value": 2025},
                "start_season": {"value": 2003},
                "num_features": {"distribution": "int_uniform", "min": 4, "max": 100},
                "data_loader": {"value": "season_average_ensemble"},
            },
        },
    },
}

In [ ]:
wandb_sweep(sweep_config, sweep_catboost_ensemble, run_count=max_runs, project="catboost")

## Submission from Best Model

In [3]:
from src.dataloaders.ensemble import EnsembleSeasonAverageDataLoader
from src.experiments import DefaultTracker
from src.models.catboost import EnsemblCatBoostModel, CatBoostHyperparamConfig
from src.experiments.config import RunConfig
from src.submissions import generate_matchups, create_submission

In [4]:
run = wandb.Api().run("aicomp-mmlm/catboost/7q665lti")
config = run.config
config

{'run_config': {'data_loader': 'season_average_ensemble',
  'num_features': 16,
  'start_season': 2003,
  'valid_season': 2025},
 'catboost_config': {'depth': 3,
  'subsample': 0.95413947576616,
  'iterations': 368,
  'grow_policy': 'Lossguide',
  'l2_leaf_reg': 0.2595120972690723,
  'border_count': 64,
  'learning_rate': 0.004759909975424424,
  'random_strength': 0.5596352030582419,
  'min_data_in_leaf': 6,
  'bagging_temperature': 2.7777687509109694}}

In [5]:
run_config = RunConfig(**config.get("run_config", {}))
dataloader = EnsembleSeasonAverageDataLoader(run_config.num_features)
run_config

RunConfig(num_features=16, valid_season=2025, start_season=2003, data_loader='season_average_ensemble')

In [6]:
hyperparameters = CatBoostHyperparamConfig(**config.get("catboost_config", {}))
hyperparameters

CatBoostHyperparamConfig(iterations=368, learning_rate=0.004759909975424424, depth=3, l2_leaf_reg=0.2595120972690723, random_strength=0.5596352030582419, bagging_temperature=2.7777687509109694, subsample=0.95413947576616, task_type='CPU', thread_count=-1, border_count=64, grow_policy='Lossguide', min_data_in_leaf=6, random_seed=42, verbose=0, allow_writing_files=False, loss_function='RMSE')

In [7]:
model = EnsemblCatBoostModel(dataloader, hyperparameters, None, DefaultTracker({}))

In [8]:
season = 2025
create_submission(season=season, model=model, filename=f"submission_catboost_ensemble_{season}.csv", fit=True)

metrics: {'train_brier_ensemble': np.float64(0.15795722215112684)}, step: 2003
metrics: {'valid_brier_ensemble': np.float64(0.18123514961152803)}, step: 2003
metrics: {'train_brier_ensemble': np.float64(0.15792410471139245)}, step: 2004
metrics: {'valid_brier_ensemble': np.float64(0.1772794125798597)}, step: 2004
metrics: {'train_brier_ensemble': np.float64(0.15805386516686507)}, step: 2005
metrics: {'valid_brier_ensemble': np.float64(0.17214810568176575)}, step: 2005
metrics: {'train_brier_ensemble': np.float64(0.15745312647073212)}, step: 2006
metrics: {'valid_brier_ensemble': np.float64(0.19453641174260614)}, step: 2006
metrics: {'train_brier_ensemble': np.float64(0.15819038481294162)}, step: 2007
metrics: {'valid_brier_ensemble': np.float64(0.16099215421780216)}, step: 2007
metrics: {'train_brier_ensemble': np.float64(0.15889452181179517)}, step: 2008
metrics: {'valid_brier_ensemble': np.float64(0.1530275042887964)}, step: 2008
metrics: {'train_brier_ensemble': np.float64(0.1582037

WindowsPath('D:/git/code/submissions/submission_catboost_ensemble_2025.csv')

## Sweep with Default Features

In [3]:
max_runs = 100
sweep_config = {
    "name": "CatBoost Ensemble (Default Features)",
    "method": "bayes",
    "metric": {"name": "cv_brier", "goal": "minimize"},
    "parameters": {
        "catboost_config": {
            "parameters": {
                "iterations": {"distribution": "int_uniform", "min": 200, "max": 1000},
                "learning_rate": {"distribution": "log_uniform_values", "min": 0.001, "max": 0.3},
                "depth": {"distribution": "int_uniform", "min": 3, "max": 10},
                "l2_leaf_reg": {"distribution": "log_uniform_values", "min": 0.1, "max": 10.0},
                "random_strength": {"distribution": "uniform", "min": 0.0, "max": 5.0},
                "bagging_temperature": {"distribution": "uniform", "min": 0.0, "max": 5.0},
                "subsample": {"distribution": "uniform", "min": 0.5, "max": 1.0},
                "border_count": {"values": [32, 64, 128, 254]},
                "grow_policy": {"values": ["SymmetricTree", "Lossguide", "Depthwise"]},
                "min_data_in_leaf": {"distribution": "int_uniform", "min": 1, "max": 10},
            },
        },
        "run_config": {
            "parameters": {
                "valid_season": {"value": 2025},
                "start_season": {"value": 2003},
                "num_features": {"value": 0},
                "data_loader": {"value": "season_average_ensemble"},
            },
        },
    },
}

In [ ]:
wandb_sweep(sweep_config, sweep_catboost_ensemble, run_count=max_runs, project="catboost")

### Submission from Best Model

In [9]:
from src.dataloaders.ensemble import EnsembleSeasonAverageDataLoader
from src.experiments import DefaultTracker
from src.models.catboost import EnsemblCatBoostModel, CatBoostHyperparamConfig
from src.experiments.config import RunConfig
from src.submissions import generate_matchups, create_submission

In [10]:
run = wandb.Api().run("aicomp-mmlm/catboost/wzg87cad")
config = run.config
config

{'run_config': {'data_loader': 'season_average_ensemble',
  'num_features': 0,
  'start_season': 2003,
  'valid_season': 2025},
 'catboost_config': {'depth': 10,
  'subsample': 0.7479447919933935,
  'iterations': 471,
  'grow_policy': 'Lossguide',
  'l2_leaf_reg': 4.386933454149256,
  'border_count': 254,
  'learning_rate': 0.009079256970292236,
  'random_strength': 3.3616455655796056,
  'min_data_in_leaf': 9,
  'bagging_temperature': 2.4102871905044725}}

In [11]:
run_config = RunConfig(**config.get("run_config", {}))
dataloader = EnsembleSeasonAverageDataLoader(run_config.num_features)
run_config

RunConfig(num_features=0, valid_season=2025, start_season=2003, data_loader='season_average_ensemble')

In [12]:
hyperparameters = CatBoostHyperparamConfig(**config.get("catboost_config", {}))
hyperparameters

CatBoostHyperparamConfig(iterations=471, learning_rate=0.009079256970292236, depth=10, l2_leaf_reg=4.386933454149256, random_strength=3.3616455655796056, bagging_temperature=2.4102871905044725, subsample=0.7479447919933935, task_type='CPU', thread_count=-1, border_count=254, grow_policy='Lossguide', min_data_in_leaf=9, random_seed=42, verbose=0, allow_writing_files=False, loss_function='RMSE')

In [13]:
model = EnsemblCatBoostModel(dataloader, hyperparameters, None, DefaultTracker({}))

In [14]:
season = 2025
create_submission(
    season=season, model=model, filename=f"submission_catboost_ensemble_default_features_{season}.csv", fit=True
)

metrics: {'train_brier_ensemble': np.float64(0.14015332906730332)}, step: 2003
metrics: {'valid_brier_ensemble': np.float64(0.18465585525749545)}, step: 2003
metrics: {'train_brier_ensemble': np.float64(0.14107952193921078)}, step: 2004
metrics: {'valid_brier_ensemble': np.float64(0.1743981188654234)}, step: 2004
metrics: {'train_brier_ensemble': np.float64(0.14036572525468902)}, step: 2005
metrics: {'valid_brier_ensemble': np.float64(0.16957621497743364)}, step: 2005
metrics: {'train_brier_ensemble': np.float64(0.1401214891848669)}, step: 2006
metrics: {'valid_brier_ensemble': np.float64(0.19309429455023602)}, step: 2006
metrics: {'train_brier_ensemble': np.float64(0.14146050254235823)}, step: 2007
metrics: {'valid_brier_ensemble': np.float64(0.1513682857240906)}, step: 2007
metrics: {'train_brier_ensemble': np.float64(0.1416799378922536)}, step: 2008
metrics: {'valid_brier_ensemble': np.float64(0.15504446539543507)}, step: 2008
metrics: {'train_brier_ensemble': np.float64(0.141064224

WindowsPath('D:/git/code/submissions/submission_catboost_ensemble_default_features_2025.csv')